# ⚠️ Exemple à ne pas suivre : inférence couplée à l'entraînement

Ce notebook recharge le modèle entraîné, mais **doit relire `train.csv` et refaire `preprocessor.fit_transform(train)`** pour préparer les données de test.
Le prétraitement apprend en effet des valeurs sur le jeu d'entraînement (âge moyen par titre et par classe, colonnes à encoder, tranches de numéros de cabine) : sans lui, on ne sait pas transformer une nouvelle donnée.

En production, le jeu d'entraînement n'est en général pas disponible au moment de prédire.
La bonne pratique consiste à sauvegarder **le modèle et le préprocesseur entraîné**, puis à n'appeler que `transform` à l'inférence : voir `titanic_predict.ipynb` (TP6, « Découpler entraînement et inférence »).

# Imports

In [ ]:
import os

In [ ]:
import pandas as pd

In [ ]:
from src.feature_engineering import Preprocessor

# Load data



In [ ]:
train = pd.read_csv(os.path.join('../input', 'train.csv'))
test = pd.read_csv(os.path.join('../input', 'test.csv'))

# Prepare data

In [ ]:
drop_columns = ['Name', 'SibSp', 'Parch', 'Cabin', 'Ticket', 'Ticket_Letter', 'Pclass', 'Sex', 'Embarked',
             'Ticket_Category', 'Cabin_Letter', 'Name_Title', 'Family_Size', 'PassengerId', 'Cabin_Number']
dummy_columns = ['Pclass', 'Sex', 'Embarked', 'Ticket_Category', 'Cabin_Letter', 'Name_Title', 'Family_Size']

preprocessor = Preprocessor(dummy_columns=dummy_columns, drop_columns=drop_columns)

train_processed = preprocessor.fit_transform(train)
test_processed = preprocessor.transform(test)

# Load model

In [ ]:
import pickle
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier

In [ ]:
MODELS_DIR = Path.cwd().parent / "models"

with open(MODELS_DIR / "model_rf.pkl", "rb") as file_in:
    model: RandomForestClassifier = pickle.load(file_in)

# Predict

In [ ]:
prediction_train = model.predict_proba(train_processed.iloc[:, 1:])

In [ ]:
prediction_test = model.predict_proba(test_processed)

In [ ]:
prediction_test